In [ ]:
cat > ~/bioinfo2_coaraci.yml << 'EOF'
name: bioinfo2
channels:
  - conda-forge
  - bioconda
dependencies:
  - python=3.11
  - sra-tools
  - parallel-fastq-dump
  - fastqc
  - fastp
  - multiqc
  - bwa
  - bwa-mem2
  - samtools
  - bcftools
  - htslib
  - gatk4
  - freebayes
  - beagle
  - parallel
EOF

In [ ]:
conda env create -f bioinfo2_coaraci.yml

In [ ]:
%%bash
fasterq-dump --split-files --threads 4 --progress SRRXXXXXXX/SRRXXXXXXX.sra

UsageError: %%bash is a cell magic, but the cell body is empty.


In [ ]:
fastqc -t 2 SRRXXXXXXX_1.fastq SRRXXXXXXX_2.fastq

In [ ]:
fastqc -t 4 *_1.fastq *_2.fastq

In [ ]:
fastqc -t 4 SRR{XXXXXXX,YYYYYYY,ZZZZZZZ}_{1,2}.fastq.gz

In [ ]:
multiqc *_fastqc.zip -n multiqc_3amostras -o ~/multiqc

In [ ]:
which qsub sbatch          # descobre o gerenciador
qsub -I -l select=1:ncpus=16:mem=64gb -l walltime=24:00:00   # se for PBS
# ou
srun -c 16 --mem=64G -t 24:00:00 --pty bash                   # se for SLURM

In [ ]:
sinfo -s

In [ ]:
srun -p par48-x -c 16 --mem=32G -t 06:00:00 --pty bash

In [ ]:
tmux new -s trabalho
srun -p par48-x -c 16 --mem=64G -t 24:00:00 --pty bash

In [ ]:
squeue -j $SLURM_JOB_ID -o "%L"

In [ ]:
module load miniconda3/24.7.1-gcc-12.2.0-ejsbpa4
eval "$(conda shell.bash hook)"
conda activate bioinfo2

In [ ]:
fastp -i SRRXXXXXXX_1.fastq -I SRRXXXXXXX_2.fastq \
  -o SRRXXXXXXX_trim_1.fastq.gz -O SRRXXXXXXX_trim_2.fastq.gz \
  --detect_adapter_for_pe \
  --cut_right --cut_right_window_size 4 --cut_right_mean_quality 20 \
  --length_required 50 -q 15 -u 40 \
  -j SRRXXXXXXX.fastp.json -h SRRXXXXXXX.fastp.html --thread 16

In [ ]:
# Generalizable minimal trimming: adapters, low-quality 3' tail (Q20) and very short reads (para 5mil amostras)
fastp -i "${S}_1.fastq.gz" -I "${S}_2.fastq.gz" \
  -o "${S}_trim_1.fastq.gz" -O "${S}_trim_2.fastq.gz" \
  --detect_adapter_for_pe \
  --cut_tail --cut_tail_window_size 4 --cut_tail_mean_quality 20 \
  --length_required 36 -q 15 -u 40 \
  -j "${S}.fastp.json" -h "${S}.fastp.html" --thread 16

In [ ]:
fastqc -t 2 SRRXXXXXXX_trim_1.fastq.gz SRRXXXXXXX_trim_2.fastq.gz

In [ ]:
bwa-mem2 mem -t 16 \
  -R "@RG\tID:SRRXXXXXXX\tSM:SRRXXXXXXX\tPL:ILLUMINA\tLB:SRRXXXXXXX" \
  genome.fa SRRXXXXXXX_trim_1.fastq.gz SRRXXXXXXX_trim_2.fastq.gz \
  > SRRXXXXXXX.sam

In [ ]:
samtools view -@ 8 -b -o SRRXXXXXXX.bam SRRXXXXXXX.sam

In [ ]:
samtools index SRRXXXXXXX.sorted.bam

In [ ]:
samtools flagstat SRRXXXXXXX.sorted.bam

In [ ]:
rm SRRXXXXXXX.sam SRRXXXXXXX.bam

In [ ]:
gatk --java-options "-Xmx48g" MarkDuplicates \
  -I SRRXXXXXXX.sorted.bam \
  -O SRRXXXXXXX.dedup.bam \
  -M SRRXXXXXXX.dedup.metrics.txt

In [ ]:
samtools index SRRXXXXXXX.dedup.bam

In [ ]:
rm SRRXXXXXXX.sorted.bam SRRXXXXXXX.sorted.bam.bai

In [ ]:
gatk --java-options "-Xmx48g" BaseRecalibrator \
  -R genome.fa \
  -I SRRXXXXXXX.dedup.bam \
  --known-sites gatk_resources/Homo_sapiens_assembly38.dbsnp138.vcf.gz \
  --known-sites gatk_resources/Mills_and_1000G_gold_standard.indels.hg38.vcf.gz \
  --known-sites gatk_resources/1000G_phase1.snps.high_confidence.hg38.vcf.gz \
  -O SRRXXXXXXX.recal.table

In [ ]:
gatk --java-options "-Xmx48g" ApplyBQSR \
  -R genome.fa \
  -I SRRXXXXXXX.dedup.bam \
  --bqsr-recal-file SRRXXXXXXX.recal.table \
  -O SRRXXXXXXX.bqsr.bam

In [ ]:
gatk --java-options "-Xmx48g" HaplotypeCaller \
  -R genome.fa \
  -I SRRXXXXXXX.bqsr.bam \
  -O SRRXXXXXXX.g.vcf.gz \
  -ERC GVCF \
  --native-pair-hmm-threads 16

In [ ]:
gatk --java-options "-Xmx48g" GenotypeGVCFs \
  -R genome.fa \
  -V SRRXXXXXXX.g.vcf.gz \
  -O SRRXXXXXXX.vcf.gz

In [ ]:
gatk --java-options "-Xmx48g" CombineGVCFs \
  -R genome.fa \
  -V SRRXXXXXXX.g.vcf.gz \
  -V SRRXXXXXXX.g.vcf.gz \
  -V SRRXXXXXXX.g.vcf.gz \
  -O AD_WXS_YES_3amostras.g.vcf.gz

# Sair do tmux sem interromper nada
### Aperte Ctrl+B, solte, e depois aperte D
Um detalhe: o tmux fica no nó de login. Se esse nó for reiniciado por manutenção (raro, mas acontece), a sessão se perde. Para etapas muito longas, o sbatch continua sendo a opção mais robusta.


In [ ]:
mkdir -p /path/to/your/wxs_joint
cd /path/to/your/wxs_joint

cat > joint_genotyping.sh << 'EOF'
#!/bin/bash
# Uso: bash joint_genotyping.sh AD   ou   bash joint_genotyping.sh DIA
set -euo pipefail
D=$1
REF=/path/to/your/ref_gatk_hg38/genome.fa
IN=/path/to/your
OUT=/path/to/your/wxs_joint
d=$(echo $D | tr A-Z a-z)

echo "[$(date)] $D: GATK $(gatk --version 2>/dev/null | grep -m1 -o 'v[0-9.]*')"

echo "[$(date)] $D: CombineGVCFs (casos + controles)"
gatk --java-options "-Xmx16g" CombineGVCFs -R $REF \
    -V $IN/20_vcf_${d}_wxs_yes/${D}_WXS_YES_combined.g.vcf.gz \
    -V $IN/20_vcf_${d}_wxs_no/${D}_WXS_NO_combined.g.vcf.gz \
    -O $OUT/${D}_WXS_all_combined.g.vcf.gz \
    --tmp-dir $OUT

echo "[$(date)] $D: GenotypeGVCFs (40 amostras juntas)"
gatk --java-options "-Xmx16g" GenotypeGVCFs -R $REF \
    -V $OUT/${D}_WXS_all_combined.g.vcf.gz \
    -O $OUT/${D}_WXS_all_joint_genotyped.vcf.gz \
    --tmp-dir $OUT

echo "[$(date)] $D: concluído -- $(bcftools query -l $OUT/${D}_WXS_all_joint_genotyped.vcf.gz | wc -l) amostras"
EOF

nohup bash joint_genotyping.sh AD  > joint_AD.log  2>&1 &
nohup bash joint_genotyping.sh DIA > joint_DIA.log 2>&1 &